In [4]:
import os
from decouple import Config, RepositoryEnv
import psycopg2
import pandas as pd

env_path = os.path.join(os.getcwd(), '..', '.env')
print("Looking for .env at:", os.path.abspath(env_path))
print("Exists?", os.path.exists(env_path))

cfg = Config(RepositoryEnv(env_path))

conn = psycopg2.connect(
    dbname=cfg('DB_NAME'),
    user=cfg('DB_USER'),
    password=cfg('DB_PASSWORD'),
    host=cfg('DB_HOST'),
    port=cfg('DB_PORT')
)
print("Connected successfully!")

Looking for .env at: C:\Users\Swechhya Baral\OneDrive\Documents\Python\academic-analytics\.env
Exists? True
Connected successfully!


In [5]:
query = """
SELECT 
    s.id AS student_id,
    AVG(g.score) AS avg_score,
    COUNT(CASE WHEN a.present THEN 1 END)::float / NULLIF(COUNT(a.id), 0) * 100 AS attendance_pct
FROM academics_student s
JOIN academics_enrollment e ON e.student_id = s.id
LEFT JOIN academics_grade g ON g.enrollment_id = e.id
LEFT JOIN academics_attendance a ON a.enrollment_id = e.id
GROUP BY s.id
"""

df = pd.read_sql(query, conn)
df = df.fillna(0)
df.head()

C:\Users\Swechhya Baral\AppData\Local\Temp\ipykernel_15244\3932316965.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,student_id,avg_score,attendance_pct
0,22,57.815000,80.000000
1,19,55.618333,83.333333
2,29,52.401667,50.000000
3,10,72.916667,80.000000
4,35,59.986667,73.333333


In [6]:
df['at_risk'] = ((df['avg_score'] < 60) | (df['attendance_pct'] < 75)).astype(int)
df.head()

,student_id,avg_score,attendance_pct,at_risk
0,22,57.815000,80.000000,1
1,19,55.618333,83.333333,1
2,29,52.401667,50.000000,1
3,10,72.916667,80.000000,0
4,35,59.986667,73.333333,1


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df[['avg_score', 'attendance_pct']]
y = df['at_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 1.0


In [8]:
import joblib
joblib.dump(model, 'risk_model.pkl')

['risk_model.pkl']

In [9]:
import os
print(os.listdir('.'))
print("File size:", os.path.getsize('risk_model.pkl'), "bytes")

['.ipynb_checkpoints', 'risk_model.pkl', 'train_model.ipynb']
File size: 1151 bytes


In [10]:
import joblib
loaded_model = joblib.load('risk_model.pkl')
test_prediction = loaded_model.predict([[55, 60]])  # example: avg_score=55, attendance=60%
print("Test prediction:", test_prediction)

Test prediction: [1]


C:\Users\Swechhya Baral\OneDrive\Documents\Python\academic-analytics\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [12]:
conn.close()